[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/09_data_aggregation/09_1_Meet_Gapminder.ipynb)

# 09.1: Meet Gapminder

In module 06 you met the main groupby tools on the Titanic data: the one-line pattern `groupby(group_col)[value_col].mean()`, several statistics at once with `.agg()`, grouping by two keys, and arranging a result into a grid with `pivot_table()`. This module returns to those tools in more depth, including the index structure that a two-key groupby produces. It also adds tools for questions they leave unanswered: what if you want the group result added back to every individual row rather than collapsed into a summary? What if you want to keep or drop whole groups based on something computed across the group?

This module works through each of those cases. The dataset we use throughout is Gapminder, a curated collection of country-level data assembled by the Gapminder Foundation to track human development over time.

The setup is the same stack you have used since module 07: pandas for the data, seaborn and matplotlib for charts, and one `sns.set_theme()` call that styles every chart in this module. The one new detail is in the load itself: Gapminder is published as a tab-separated file rather than a comma-separated one, which is why `read_csv()` needs `sep="\t"`.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")

url = "https://raw.githubusercontent.com/jennybc/gapminder/main/inst/extdata/gapminder.tsv"
df = pd.read_csv(url, sep="\t")
df.head()

## What this dataset contains

Before grouping anything, you need to know what each row represents. Run `df.info()` to see the column names, their types, and whether any values are missing. For this dataset, the shape itself is a clue: if you know there are 142 countries and 12 time points, you can already predict how many rows a clean panel dataset should have.

In [ ]:
df.info()

Six columns, 1,704 rows, no missing values. The columns are:

- `country`: the country name (142 unique countries)
- `continent`: one of Africa, Americas, Asia, Europe, Oceania
- `year`: one of 12 values from 1952 to 2007 in 5-year steps
- `lifeExp`: life expectancy at birth, in years
- `pop`: total population
- `gdpPercap`: GDP per capita in inflation-adjusted US dollars

This is **panel data**: each country appears once per year, so the dataset has 142 countries × 12 years = 1,704 rows. The unique identifier for each row is the `(country, year)` pair.

In [ ]:
print("Countries:", df["country"].nunique())
print("Continents:", df["continent"].value_counts().to_dict())
print("Years:", sorted(df["year"].unique()))

The continent counts above are rows, so dividing each by the 12 years gives the number of countries. Africa has the most (624 rows, so 52 countries), followed by Asia (33), Europe (30), and the Americas (25). Oceania has only 2 (Australia and New Zealand). Oceania summaries represent two countries, not a whole region in the same sense as the others, and that will matter when we compare continent-level statistics.

The 12 years from 1952 to 2007 give us a 55-year window that spans decolonization, the Cold War, the AIDS epidemic in sub-Saharan Africa, and rapid economic development in East Asia.

## A first look at the numerics

The three numeric columns measure very different things on very different scales. Before grouping by anything, look at the ranges: knowing the minimum and maximum of each column tells you whether extreme values exist and whether any aggregation function you compute will be pulled by outliers.

In [ ]:
df[["lifeExp", "pop", "gdpPercap"]].describe().round(1)

Three things stand out. First, life expectancy ranges from 23.6 years to 82.6 years, a span of nearly 60 years within the same dataset. The minimum is Rwanda in 1992, during the civil war that led up to the 1994 genocide, and the maximum is Japan in 2007. Second, population ranges from 60,000 to 1.3 billion, a range of four orders of magnitude that will matter if we ever weight aggregations by population. Third, GDP per capita has a mean of $7,215 but a median of only $3,532. The mean sitting far above the median is the sign of right skew: a relatively small number of very high values pull the average up considerably.

These statistics describe all 1,704 rows at once, mixing every country and every year together. They are useful for seeing the ranges, but most of the questions we want to ask in this module are about groups, such as continents or years, so the next step is to break the data into those groups.

## A quick review of single-key groupby

You already know how to answer a question like "what is the mean life expectancy in each continent?" using the groupby you learned in module 06. Let's run it once to confirm the baseline and establish what the rest of this module builds on.

In [ ]:
# Filter to 2007 for a snapshot comparison
df2007 = df[df["year"] == 2007].copy()

df2007.groupby("continent")["lifeExp"].mean().round(1).sort_values(ascending=False)

In 2007, Oceania has the highest mean life expectancy, about 81 years, followed by Europe at about 78. Africa is lowest at about 55. This one line answers the question, but it raises others: how much variation is there within each continent? How did these numbers change over time? How does each individual country compare to its continental peers?

A single-key groupby with a single aggregation function cannot answer those follow-up questions. Variation within a continent needs more than one statistic per group (09.3), change over time needs a second grouping key (09.2), and comparing a country to its continent needs the group result placed back on each row (09.4).

## What the data looks like overall

The three numeric columns have very different distributions. Visualizing them before computing any grouped statistics helps you know what to expect and spot anything surprising. The histograms below use only the 2007 rows, so each country appears once. Unusual values in this dataset, such as the very low life expectancies in some countries, are usually real events (wars, famines, epidemics) rather than data errors, so it is worth looking up the history before treating a value as a mistake.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

sns.histplot(data=df2007, x="lifeExp", bins=20, ax=axes[0])
axes[0].set_title("Life expectancy (2007)")
axes[0].set_xlabel("Years")

sns.histplot(data=df2007, x="gdpPercap", bins=20, ax=axes[1])
axes[1].set_title("GDP per capita (2007)")
axes[1].set_xlabel("USD (inflation-adjusted)")

sns.histplot(data=df2007, x="pop", bins=20, log_scale=True, ax=axes[2])
axes[2].set_title("Population (2007, log scale)")
axes[2].set_xlabel("Population")

plt.tight_layout()

Life expectancy has a bimodal shape in 2007: a low, fairly flat stretch of countries from about 40 to the mid 60s (mostly in Africa, with many sub-Saharan countries affected by the AIDS epidemic) and a taller cluster from about 70 into the low 80s. GDP per capita is strongly right-skewed: most countries are clustered at low values, with a long tail of wealthy nations. Population requires a log scale to be legible at all, spanning from tiny island nations to China and India.

With a bimodal life expectancy distribution and a strongly skewed GDP distribution, a single mean computed across all countries falls between the clusters and does not describe any particular country well. Grouping by continent or year gives statistics that describe more similar sets of countries, which is why the rest of this module is about grouped summaries.

## What's next

The single-key groupby you already know answers one question at a time about one grouping variable. Notebook 09.2 returns to multi-key groupby, which you first saw in module 06: grouping by continent AND year simultaneously to track how each continent has changed over time. It looks closely at the structure that kind of groupby produces, the MultiIndex, and at the `unstack()` method for reshaping it.